# 📡 Cálculo de Frequências de Ressonância — CSRR em FR-4

**Ressonadores de Anel Fendido Complementar (CSRR)** gravados no plano de terra de uma microstrip sobre substrato FR-4.

---

## Parâmetros da simulação

| Parâmetro | Símbolo | Variável? |
|---|---|---|
| Unidade base | u | ✅ **Variável** |
| Gap entre elementos | g | ✅ **Variável** |
| Período da célula unitária | P | ✅ **Variável** |
| Espessura do substrato | h | Constante |
| Largura da trilha | w | Constante |
| Espessura do metal Cu | t | Constante |
| Permissividade relativa FR-4 | εr | Constante |
| Tangente de perda FR-4 | tan δ | Constante |
| Condutividade do cobre | σ | Constante |

---

> **Como usar:** Execute todas as células em ordem (`Runtime → Run all`).  
> Para variar os parâmetros, edite **apenas a célula de Parâmetros Variáveis** e execute novamente.

In [ ]:
# ============================================================
#  Importações
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings('ignore')

print('✅ Bibliotecas carregadas com sucesso.')

## ▶ Parâmetros Variáveis
**Edite apenas esta célula para mudar a simulação.**

In [ ]:
# ============================================================
#  ▶▶ ALTERE APENAS ESTES TRÊS VALORES (em mm) ◀◀
# ============================================================

u = 1.0    # Unidade base                [mm]
g = 0.5    # Gap entre elementos         [mm]
P = 7.0    # Período da célula unitária  [mm]

# ============================================================

print(f'  Unidade base (u) = {u} mm')
print(f'  Gap         (g) = {g} mm')
print(f'  Período     (P) = {P} mm')

## Constantes Fixas

In [ ]:
# ============================================================
#  Constantes do substrato e do condutor — NÃO ALTERE
# ============================================================

# Geometria
h     = 1.20     # Espessura do substrato FR-4   [mm]
w     = 0.50     # Largura da trilha microstrip  [mm]
t     = 0.035    # Espessura do metal (Cu)       [mm]

# Propriedades do material
eps_r    = 4.4       # Permissividade relativa FR-4   [adimensional]
tan_d    = 0.020     # Tangente de perda FR-4         [adimensional]
sigma_cu = 5.8e7     # Condutividade do cobre         [S/m]

# Constantes físicas (SI)
c0   = 299_792_458.0        # Velocidade da luz no vácuo     [m/s]
eps0 = 8.854_187_817e-12    # Permissividade do vácuo        [F/m]
mu0  = 4.0 * np.pi * 1e-7  # Permeabilidade do vácuo        [H/m]

print('✅ Constantes definidas.')

## Parâmetros da Microstrip

Calculados pela fórmula de Schneider (1969) / Hammerstad-Jensen para permissividade efetiva e pela fórmula de Wheeler (1977) para impedância característica.

In [ ]:
# ============================================================
#  Microstrip — Permissividade Efetiva (Schneider 1969)
# ============================================================
wh = w / h   # razão w/h

if wh <= 1:
    F = 1.0 / np.sqrt(1.0 + 12.0 * h / w) + 0.04 * (1.0 - w / h) ** 2
else:
    F = 1.0 / np.sqrt(1.0 + 12.0 * h / w)

eps_eff = (eps_r + 1.0) / 2.0 + (eps_r - 1.0) / 2.0 * F

# ============================================================
#  Microstrip — Impedância Característica Z0 (Wheeler 1977)
# ============================================================
if wh <= 1:
    Z0 = (60.0 / np.sqrt(eps_eff)) * np.log(8.0 * h / w + w / (4.0 * h))
else:
    Z0 = (120.0 * np.pi) / (np.sqrt(eps_eff) * (wh + 1.393 + 0.667 * np.log(wh + 1.444)))

# Permissividade efetiva para fenda (slot) no plano de terra
# Pelo princípio de Babinet: média aritmética ar/substrato
eps_eff_slot = (1.0 + eps_r) / 2.0

print(f'  w/h                           = {wh:.4f}')
print(f'  Permissividade efetiva  εeff  = {eps_eff:.4f}')
print(f'  εeff do slot (Babinet)        = {eps_eff_slot:.4f}')
print(f'  Impedância característica Z0  = {Z0:.2f} Ω')

## Geometria do CSRR

Estrutura considerada: **CSRR de duplo anel quadrado** gravado no plano de terra abaixo da microstrip.

```
┌─────────────────────────────── P ───────────────────────────────┐
│  g  ┌── a = P-2g ──────────────────────────────────────────┐  g │
│     │  u ┌──────────────────────────────────────────────┐  │    │
│     │    │  g ┌────────────────────────────────────┐  g │  │    │
│     │    │    │  u ┌──────────────────────────┐  u │    │  │    │
│     │    │    │    │      (interior)           │    │    │  │    │
│     │    │    │    └──────────────────────────┘    │    │  │    │
│     │    │    └────────────────────────────────────┘    │  │    │
│     │    └──────────────────────────────────────────────┘  │    │
│     └──────────────────────────────────────────────────────┘    │
└─────────────────────────────────────────────────────────────────┘
  Anel externo: lado = a = P - 2g        Largura do anel: c = u
  Gap entre anéis: g                     Período da célula: P
```


In [ ]:
# ============================================================
#  Geometria do CSRR de duplo anel quadrado
# ============================================================

# Lado externo do anel mais externo (com margem de gap em cada lado)
a = P - 2.0 * g          # [mm]

# Largura de cada anel metálico = unidade base
c_w = u                  # [mm]

# Lado interno do anel externo
a_int_ext  = a - 2.0 * c_w          # [mm]

# Lado externo do anel interno (separação = g entre os dois anéis)
a_ext_int  = a_int_ext - 2.0 * g    # [mm]

# Lado interno do anel interno
a_int_int  = a_ext_int - 2.0 * c_w  # [mm]

# Validação de geometria
assert a > 0,            f'Erro: a = P - 2g = {a:.3f} mm ≤ 0. Aumente P ou reduza g.'
assert a_int_ext > 0,    f'Erro: anel externo muito estreito (a_int_ext = {a_int_ext:.3f} mm). Reduza u ou g.'
assert a_ext_int > 0,    f'Erro: anel interno não cabe (a_ext_int = {a_ext_int:.3f} mm). Reduza u ou g.'
assert a_int_int > 0,    f'Erro: interior do anel interno negativo (a_int_int = {a_int_int:.3f} mm). Reduza u ou g.'

# Perímetros (mm)
P_mais_ext = 4.0 * a           # Perímetro mais externo da estrutura
P_mais_int = 4.0 * a_int_int   # Perímetro mais interno da estrutura
P_mean     = (P_mais_ext + P_mais_int) / 2.0  # Perímetro médio

print(f'  Lado externo do CSRR       a  = {a:.3f} mm')
print(f'  Largura do anel            c  = {c_w:.3f} mm')
print(f'  Lado interno (anel ext.)      = {a_int_ext:.3f} mm')
print(f'  Lado externo (anel int.)      = {a_ext_int:.3f} mm')
print(f'  Lado interno (anel int.)      = {a_int_int:.3f} mm')
print(f'  Perímetro mais externo        = {P_mais_ext:.3f} mm')
print(f'  Perímetro mais interno        = {P_mais_int:.3f} mm')
print(f'  Perímetro médio         l_av  = {P_mean:.3f} mm')

## Cálculo das Frequências de Ressonância

São utilizados **dois modelos complementares**:

### Modelo 1 — Ressonador λ/2 (perímetro médio)
O CSRR ressoa quando o comprimento de onda efetivo corresponde ao dobro do perímetro médio:
$$f_0 = \frac{c_0}{2 \, l_{\text{av}} \, \sqrt{\varepsilon_{\text{eff, slot}}}}$$

### Modelo 2 — Circuito LC equivalente
O CSRR é modelado como um circuito ressonante LC:
$$f_0 = \frac{1}{2\pi\sqrt{L_c \, C_c}}$$

onde $L_c$ é a indutância do anel (fórmula de Neumann) e $C_c$ é a capacitância do gap (modelo de placa paralela por borda).

In [ ]:
# ============================================================
#  Modelo 1: Ressonador λ/2  —  f0 = c0 / (2 · l_av · √εeff_slot)
# ============================================================
l_av_m = P_mean * 1e-3   # perímetro médio em metros

f0_lam_half     = c0 / (2.0 * l_av_m * np.sqrt(eps_eff_slot))   # Hz
f0_lam_half_GHz = f0_lam_half / 1e9                              # GHz

# Harmônicas (n = 1 … 5)
n_max  = 5
f_harm = [n * f0_lam_half_GHz for n in range(1, n_max + 1)]

# ============================================================
#  Modelo 2: Circuito LC equivalente
# ============================================================
c_w_m = c_w * 1e-3   # largura do anel [m]
g_m   = g   * 1e-3   # gap [m]

# Indutância: fórmula de Neumann simplificada para malha plana quadrada
#   L ≈ (μ₀ / 2π) · l_av · [ln(2·l_av / (c_w + g)) − 0.774]
arg_log = 2.0 * l_av_m / (c_w_m + g_m)
assert arg_log > np.e ** 0.774, (
    f"Geometria inválida para o modelo de indutância: "
    f"l_av/(c_w+g) = {l_av_m/(c_w_m+g_m):.3f} é muito pequeno. "
    "Aumente P ou reduza u e g.")
L_c = (mu0 / (2.0 * np.pi)) * l_av_m * (np.log(arg_log) - 0.774)

# Capacitância: gap do anel fendido
# Modelo de capacitor de borda: c_w × h / g
# (h = espessura do substrato, define a escala das linhas de campo franja)
#   C ≈ ε₀ · εeff_slot · c_w · h / g  (dois gaps em paralelo → ×2)
h_m = h * 1e-3
C_c = 2.0 * eps0 * eps_eff_slot * c_w_m * h_m / g_m

# Frequência de ressonância LC
f0_LC     = 1.0 / (2.0 * np.pi * np.sqrt(L_c * C_c))   # Hz
f0_LC_GHz = f0_LC / 1e9                                 # GHz

# ============================================================
#  Referência: ressonância λ/2 da microstrip de comprimento P
# ============================================================
f0_ms_GHz = c0 / (2.0 * P * 1e-3 * np.sqrt(eps_eff)) / 1e9

print('✅ Frequências calculadas.')
print(f'  Modelo λ/2   → f₀ = {f0_lam_half_GHz:.4f} GHz')
print(f'  Modelo LC    → f₀ = {f0_LC_GHz:.4f} GHz')
print(f'  Microstrip P → f₀ = {f0_ms_GHz:.4f} GHz')

## Perdas e Fator de Qualidade

In [ ]:
# ============================================================
#  Perdas e fator de qualidade @ f0 (Modelo λ/2)
# ============================================================

# Profundidade de pele (skin depth)
delta_skin = np.sqrt(1.0 / (np.pi * f0_lam_half * mu0 * sigma_cu))  # [m]

# Resistência de superfície do cobre
R_s = np.sqrt(np.pi * f0_lam_half * mu0 / sigma_cu)  # [Ω/□]

# Q dielétrico (FR-4)
Q_d = 1.0 / tan_d

# Q do condutor (estimativa microstrip — Pozar)
# Qc = (π · f · μ₀ · σ)^½ · Z0 · h / (π · Rs)
Q_c = (Z0 * h * 1e-3) / (R_s * (w * 1e-3 + 2.0 * h * 1e-3))

# Q total (Matthiessen inverso)
Q_total = 1.0 / (1.0 / Q_d + 1.0 / Q_c)

print(f'  Skin depth @ f₀            δs = {delta_skin * 1e6:.4f} μm')
print(f'  Resistência de superfície  Rs = {R_s * 1000:.4f} mΩ/□')
print(f'  Q dielétrico               Qd = {Q_d:.1f}')
print(f'  Q condutor (estimativa)    Qc = {Q_c:.1f}')
print(f'  Q total (estimativa)        Q = {Q_total:.1f}')

## Resultados Completos

In [ ]:
# ============================================================
#  Impressão formatada dos resultados
# ============================================================
SEP = '=' * 65

print(SEP)
print('     FREQUÊNCIAS DE RESSONÂNCIA — CSRR EM FR-4')
print(SEP)

print('\n📌 PARÂMETROS VARIÁVEIS:')
print(f'   Unidade base           u  = {u:>8.3f}  mm')
print(f'   Gap entre elementos    g  = {g:>8.3f}  mm')
print(f'   Período da célula      P  = {P:>8.3f}  mm')

print('\n📌 CONSTANTES DO SUBSTRATO / CONDUTOR:')
print(f'   Espessura substrato    h  = {h:>8.3f}  mm')
print(f'   Largura da trilha      w  = {w:>8.3f}  mm')
print(f'   Espessura metal Cu     t  = {t:>8.4f}  mm')
print(f'   Permissividade FR-4    εr = {eps_r:>8.1f}')
print(f'   Tangente de perda    tanδ = {tan_d:>8.4f}')
print(f'   Condutividade Cu       σ  = {sigma_cu:.2e}  S/m')

print('\n📐 PARÂMETROS DA MICROSTRIP:')
print(f'   w/h                       = {wh:.4f}')
print(f'   Permissividade efetiva εeff= {eps_eff:.4f}')
print(f'   εeff slot (Babinet)        = {eps_eff_slot:.4f}')
print(f'   Impedância caracte.   Z0  = {Z0:.2f}  Ω')

print('\n📐 GEOMETRIA DO CSRR (duplo anel quadrado):')
print(f'   Lado externo           a  = {a:.3f}  mm')
print(f'   Largura do anel        c  = {c_w:.3f}  mm')
print(f'   Gap entre anéis        g  = {g:.3f}  mm')
print(f'   Perímetro externo         = {P_mais_ext:.3f}  mm')
print(f'   Perímetro interno         = {P_mais_int:.3f}  mm')
print(f'   Perímetro médio       l_av = {P_mean:.3f}  mm')

print('\n🔔 FREQUÊNCIAS DE RESSONÂNCIA:')
print(f'\n   ▶ Modelo 1 — λ/2 (perímetro médio):')
print(f'     f₀ = {f0_lam_half_GHz:.4f} GHz  ({f0_lam_half_GHz * 1000:.2f} MHz)')

print(f'\n   ▶ Modelo 2 — Circuito LC equivalente:')
print(f'     L  = {L_c * 1e9:.4f} nH')
print(f'     C  = {C_c * 1e12:.4f} pF')
print(f'     f₀ = {f0_LC_GHz:.4f} GHz  ({f0_LC_GHz * 1000:.2f} MHz)')

print(f'\n   ▶ Ref. — Microstrip λ/2 de comprimento P:')
print(f'     f₀ = {f0_ms_GHz:.4f} GHz  ({f0_ms_GHz * 1000:.2f} MHz)')

print(f'\n   ▶ Harmônicas do Modelo λ/2 (n = 1 … {n_max}):')
print(f'   {"n":>4}  {"Frequência (GHz)":>18}  {"Comprimento de onda λ (mm)":>28}')
print(f'   {"-"*4}  {"-"*18}  {"-"*28}')
for n, fn in enumerate(f_harm, 1):
    lam_n = (c0 / (fn * 1e9)) * 1e3   # mm no ar
    print(f'   {n:>4}  {fn:>18.4f}  {lam_n:>28.4f}')

print('\n⚡ PERDAS @ f₀ (Modelo λ/2):')
print(f'   Skin depth             δs = {delta_skin * 1e6:.4f} μm')
print(f'   Resistência superficial Rs = {R_s * 1000:.4f} mΩ/□')
print(f'   Q dielétrico           Qd = {Q_d:.1f}')
print(f'   Q condutor (estimativa) Qc = {Q_c:.1f}')
print(f'   Q total (estimativa)    Q  = {Q_total:.1f}')
print(SEP)

## Visualização

In [ ]:
# ============================================================
#  Gráfico 1: Resposta em frequência estimada (Lorentziano)
#  Gráfico 2: f₀ vs Período P  (varredura paramétrica)
#  Gráfico 3: f₀ vs Unidade base u
#  Gráfico 4: f₀ vs Gap g
# ============================================================

def lorentzian_S21(f_GHz, f_poles_GHz, Q_val):
    """Resposta Lorentziana ilustrativa de S21 (dB)."""
    S21_lin = np.ones_like(f_GHz)
    for fp in f_poles_GHz:
        S21_lin *= (1.0 / (1.0 + Q_val**2 * ((f_GHz / fp) - (fp / f_GHz))**2)) ** 0.5
    return 20.0 * np.log10(np.maximum(S21_lin, 1e-9))


def f0_para_P(P_val, u_val, g_val, eps_eff_slot_val):
    """Calcula f₀ (GHz, modelo λ/2) para um dado P (mm)."""
    a_v = P_val - 2.0 * g_val
    if a_v <= 0:
        return np.nan
    ai_ext = a_v - 2.0 * u_val
    if ai_ext <= 0:
        return np.nan
    ae_int = ai_ext - 2.0 * g_val
    if ae_int <= 0:
        return np.nan
    ai_int = ae_int - 2.0 * u_val
    if ai_int <= 0:
        return np.nan
    Pm = (4.0 * a_v + 4.0 * ai_int) / 2.0 * 1e-3   # m
    return c0 / (2.0 * Pm * np.sqrt(eps_eff_slot_val)) / 1e9


# Faixa de frequência para o espectro
f_max_plot = max(f_harm[-1] * 1.25, f0_lam_half_GHz * 2.0)
f_vec = np.linspace(0.05, f_max_plot, 3000)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    f'CSRR em FR-4 — u={u} mm | g={g} mm | P={P} mm',
    fontsize=14, fontweight='bold'
)

# ── Gráfico 1: Resposta S21 ──────────────────────────────────
ax1 = axes[0, 0]
S21 = lorentzian_S21(f_vec, f_harm, Q_total)
ax1.plot(f_vec, S21, 'b-', linewidth=1.8, label='|S₂₁| estimado')
colors = plt.cm.Reds(np.linspace(0.4, 0.9, n_max))
for n, fn in enumerate(f_harm, 1):
    ax1.axvline(x=fn, color=colors[n - 1], linestyle='--', alpha=0.8,
                label=f'f{n}={fn:.2f} GHz' if n <= 3 else None)
ax1.set_xlabel('Frequência (GHz)', fontsize=11)
ax1.set_ylabel('|S₂₁| (dB)', fontsize=11)
ax1.set_title('Resposta em Frequência Estimada\n(Modelo Lorentziano ilustrativo)', fontsize=10)
ax1.legend(fontsize=8, loc='lower left')
ax1.grid(True, alpha=0.35)
ax1.set_xlim([0, f_max_plot])
ax1.set_ylim([-35, 2])

# ── Gráfico 2: f₀ vs Período P ───────────────────────────────
ax2 = axes[0, 1]
P_min_valid = 4.0 * g + 4.0 * u + 0.5   # mínimo geométrico
P_sweep = np.linspace(P_min_valid, 20.0, 300)
f_vs_P = [f0_para_P(Pv, u, g, eps_eff_slot) for Pv in P_sweep]
ax2.plot(P_sweep, f_vs_P, 'g-', linewidth=2, label='f₀ vs P')
if not np.isnan(f0_lam_half_GHz):
    ax2.scatter([P], [f0_lam_half_GHz], color='red', s=100, zorder=5,
                label=f'Ponto atual\n({P} mm, {f0_lam_half_GHz:.2f} GHz)')
    ax2.axvline(x=P, color='orange', linestyle='--', linewidth=1.5, alpha=0.7)
    ax2.axhline(y=f0_lam_half_GHz, color='red', linestyle=':', alpha=0.5)
ax2.set_xlabel('Período P (mm)', fontsize=11)
ax2.set_ylabel('f₀ (GHz)', fontsize=11)
ax2.set_title(f'f₀ vs Período  (u={u} mm, g={g} mm)', fontsize=10)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.35)

# ── Gráfico 3: f₀ vs Unidade base u ──────────────────────────
ax3 = axes[1, 0]
u_max = (P - 2.0 * g) / 4.0 - g / 2.0 - 0.05
u_sweep = np.linspace(0.1, max(u_max, 0.2), 300)
f_vs_u = [f0_para_P(P, uv, g, eps_eff_slot) for uv in u_sweep]
ax3.plot(u_sweep, f_vs_u, 'm-', linewidth=2, label='f₀ vs u')
if not np.isnan(f0_lam_half_GHz):
    ax3.scatter([u], [f0_lam_half_GHz], color='red', s=100, zorder=5,
                label=f'Ponto atual\n({u} mm, {f0_lam_half_GHz:.2f} GHz)')
    ax3.axvline(x=u, color='orange', linestyle='--', linewidth=1.5, alpha=0.7)
    ax3.axhline(y=f0_lam_half_GHz, color='red', linestyle=':', alpha=0.5)
ax3.set_xlabel('Unidade base u (mm)', fontsize=11)
ax3.set_ylabel('f₀ (GHz)', fontsize=11)
ax3.set_title(f'f₀ vs Unidade Base  (P={P} mm, g={g} mm)', fontsize=10)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.35)

# ── Gráfico 4: f₀ vs Gap g ───────────────────────────────────
ax4 = axes[1, 1]
g_max = (P - 4.0 * u) / 6.0 - 0.05
g_sweep = np.linspace(0.05, max(g_max, 0.1), 300)
f_vs_g = [f0_para_P(P, u, gv, eps_eff_slot) for gv in g_sweep]
ax4.plot(g_sweep, f_vs_g, 'c-', linewidth=2, label='f₀ vs g')
if not np.isnan(f0_lam_half_GHz):
    ax4.scatter([g], [f0_lam_half_GHz], color='red', s=100, zorder=5,
                label=f'Ponto atual\n({g} mm, {f0_lam_half_GHz:.2f} GHz)')
    ax4.axvline(x=g, color='orange', linestyle='--', linewidth=1.5, alpha=0.7)
    ax4.axhline(y=f0_lam_half_GHz, color='red', linestyle=':', alpha=0.5)
ax4.set_xlabel('Gap g (mm)', fontsize=11)
ax4.set_ylabel('f₀ (GHz)', fontsize=11)
ax4.set_title(f'f₀ vs Gap  (P={P} mm, u={u} mm)', fontsize=10)
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.35)

plt.tight_layout()
plt.savefig('csrr_resonance.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nGráfico salvo como 'csrr_resonance.png'")

## Varredura Paramétrica em Tabela

Calcula f₀ para diferentes combinações de u, g e P de forma sistemática.

In [ ]:
# ============================================================
#  Tabela de varredura paramétrica
#  Altere as listas abaixo para explorar combinações
# ============================================================

u_list = [0.5, 1.0, 1.5]       # Unidade base [mm]
g_list = [0.3, 0.5, 0.8]       # Gap [mm]
P_list = [5.0, 7.0, 9.0, 11.0] # Período [mm]

header = f"{'u (mm)':>8} {'g (mm)':>8} {'P (mm)':>8} {'f₀ λ/2 (GHz)':>14} {'f₀ LC (GHz)':>13}"
print(header)
print('-' * len(header))

for u_i in u_list:
    for g_i in g_list:
        for P_i in P_list:
            # ── λ/2 ──
            f_lhalf = f0_para_P(P_i, u_i, g_i, eps_eff_slot)

            # ── LC ──
            a_i    = P_i - 2.0 * g_i
            ai_ext = a_i - 2.0 * u_i
            ae_int = ai_ext - 2.0 * g_i if ai_ext > 0 else -1
            ai_int = ae_int - 2.0 * u_i if ae_int > 0 else -1
            if ai_int > 0:
                Pm_i  = (4.0 * a_i + 4.0 * ai_int) / 2.0 * 1e-3
                cw_i  = u_i * 1e-3
                gm_i  = g_i * 1e-3
                arg   = 2.0 * Pm_i / (cw_i + gm_i)
                L_i   = (mu0 / (2.0 * np.pi)) * Pm_i * max(np.log(arg) - 0.774, 0.01)
                hm_i  = h * 1e-3
                C_i   = 2.0 * eps0 * eps_eff_slot * cw_i * hm_i / gm_i
                f_lc_i = 1.0 / (2.0 * np.pi * np.sqrt(L_i * C_i)) / 1e9
            else:
                f_lc_i = float('nan')

            f_lhalf_str = f'{f_lhalf:.4f}' if not np.isnan(f_lhalf) else '  N/A  '
            f_lc_str    = f'{f_lc_i:.4f}'  if not np.isnan(f_lc_i)  else '  N/A  '
            print(f'{u_i:>8.2f} {g_i:>8.2f} {P_i:>8.1f} {f_lhalf_str:>14} {f_lc_str:>13}')

---
## Referências

- **Schneider, M.V.** (1969). *Microstrip lines for microwave integrated circuits*. Bell System Technical Journal.
- **Wheeler, H.A.** (1977). *Transmission-line properties of a strip on a dielectric sheet on a plane*. IEEE Trans. MTT.
- **Falcone, F. et al.** (2004). *Babinet Principle Applied to the Design of Metasurfaces and Metamaterials*. Physical Review Letters.
- **Bonache, J. et al.** (2006). *Novel microstrip bandpass filters based on complementary split-ring resonators*. IEEE Trans. MTT.
- **Pozar, D.M.** (2011). *Microwave Engineering*, 4th ed. Wiley.

---
*Notebook gerado para o projeto DFTE/UFRN — Cálculo analítico de frequências de ressonância de CSRRs em substrato FR-4.*